# Deep Borehole Heat Exchanger (DBHE) for Absorption Cooling
## Onshore South Louisiana Legacy Well · Bare vs Insulated Tubing
### Solver v1 — GITT Formation Model · LiBr Single-Effect Absorption Chiller

**Physical concept:**
Cool water is injected continuously down the annulus of a legacy wellbore,
extracts geothermal heat from the formation, and returns hot to the surface.
The hot return stream drives a single-effect LiBr–water absorption chiller
that delivers cold to a building or industrial cooling load.

**Two sub-cases compared:**
- **Case A — Bare steel tubing:** Standard legacy completion. Hot upflowing
  fluid loses heat to cool downflowing annulus fluid (thermal short-circuit).
- **Case B — Insulated tubing:** Vacuum-insulated or polymer-coated tubing
  (kt_eff ≈ 0.02 W/m·K). Short-circuit suppressed; Tret maximised.

**Novel aspects vs existing DBHE literature:**
1. Real Gulf Coast legacy wellbore geometry (onshore S. Louisiana)
2. Absorption *cooling* (not heating) in a subtropical, cooling-dominated climate
3. GITT exact formulation over 20-year horizon (Rfar = 200 m)
4. Quantified performance penalty of bare vs insulated legacy completions
5. End-of-life criterion: Tret < 78°C → LiBr chiller no longer viable


In [ ]:
!pip install CoolProp -q

import numpy as np
from scipy.special import j0, j1, y0, y1
from scipy.optimize import brentq
from scipy.linalg import expm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

try:
    from CoolProp.CoolProp import PropsSI
    COOLPROP_OK = True
    print("CoolProp ✓")
except ImportError:
    COOLPROP_OK = False
    print("WARNING: CoolProp not found — pip install CoolProp")

print("Imports OK")


## 1 · Well Profile — Onshore South Louisiana Representative

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Representative onshore S. Louisiana legacy well
# Completion: 2-7/8" tubing in 7" production casing, 8.5" open hole
# Source geometry: standard API casing/tubing specifications
# ─────────────────────────────────────────────────────────────────────────────

# ── Geometry ──────────────────────────────────────────────────────────────
rt_i  = 0.03099      # 2-7/8" tubing inner radius  (2.441" ID) [m]
rt_o  = 0.03652      # 2-7/8" tubing outer radius  (2.875" OD) [m]
rca_i = 0.07975      # 7" casing inner radius       (6.276" ID, 26 lb/ft) [m]
rca_o = 0.08890      # 7" casing outer radius       (7.000" OD) [m]
rce   = 0.10795      # cement outer radius           (8.5" hole → ~8.5"/2) [m]

kca   = 45.0         # casing steel thermal conductivity [W/(m·K)]
kce   = 0.9          # Gulf Coast cement (silica-flour blend) [W/(m·K)]

# ── Sub-case conductivities ────────────────────────────────────────────────
kt_bare      = 45.0   # bare steel tubing  [W/(m·K)]
kt_insulated = 0.02   # vacuum-insulated tubing (VIT) effective k [W/(m·K)]
# kt_insulated ≈ 0.02 W/(m·K) is consistent with commercial VIT product data

# ── Formation — Gulf Coast sedimentary basin ──────────────────────────────
ke      = 2.2         # formation thermal conductivity [W/(m·K)]  (shale-dominant)
alpha_e = 1.05e-6     # thermal diffusivity [m²/s]
T_surf  = 20.0        # mean annual surface temperature, S. Louisiana [°C]
Gg      = 0.035       # geothermal gradient [K/m]  (Gulf Coast typical)

# ── Operating conditions ──────────────────────────────────────────────────
Tin     = 25.0        # injection temperature [°C]  (surface ambient + slight warming)
P_op    = 50e5        # wellbore operating pressure [Pa]  (hydrostatic at ~500m)
T_cement_limit = 165. # Class G/H cement reliability limit [°C]

# ── Well depth (primary design variable) ─────────────────────────────────
Lbh_default = 2500.   # default depth [m]  → T∞(bottom) = 20+0.035*2500 = 107.5°C

# ── Cycle / operation ─────────────────────────────────────────────────────
f_op      = 0.60      # daily operating fraction (14.4 h/day cooling load)
t_day     = 86400.    # seconds per day
t_on      = f_op * t_day    # operating seconds per day
t_off     = (1-f_op) * t_day

# ── Long-term simulation ──────────────────────────────────────────────────
n_years   = 20        # simulation horizon [years]
dt_month  = 30 * t_day   # monthly time step [s]
n_months  = n_years * 12

# ── GITT — CRITICAL: Rfar must accommodate 20-year thermal diffusion ───────
delta_20yr = np.sqrt(4 * alpha_e * n_years * 365.25 * t_day)
Rfar = max(200., 3.5 * delta_20yr)   # guaranteed < 0.1% contamination at 20 yr

# ── Numerical ─────────────────────────────────────────────────────────────
NZ    = 20     # axial cells
Neig  = 60     # GITT eigenvalues (more than TES solver — need long-τ modes)

print(f"Representative well:  Lbh = {Lbh_default:.0f} m")
print(f"BHT (undisturbed):   {T_surf + Gg*Lbh_default:.1f} °C")
print(f"20-yr thermal radius: δ = {delta_20yr:.1f} m  →  Rfar = {Rfar:.0f} m")
print(f"Daily operation:     {f_op*24:.1f} h/day  ({f_op:.0%} duty cycle)")
print(f"Simulation horizon:  {n_years} years  ({n_months} monthly steps)")
print(f"\nGeometry summary:")
print(f"  Tubing:  ID={2*rt_i*39.37:.3f}"  OD={2*rt_o*39.37:.3f}"")
print(f"  Casing:  ID={2*rca_i*39.37:.3f}"  OD={2*rca_o*39.37:.3f}"")
print(f"  Hole:    {2*rce*39.37:.2f}"")


## 2 · Geometry and Conductances

In [ ]:
# Cross-sectional areas and hydraulic diameters
Aa    = np.pi * (rca_i**2 - rt_o**2)
Atb   = np.pi * rt_i**2
Dh_a  = 2 * (rca_i - rt_o)
Dh_tb = 2 * rt_i
rlm   = (rt_o - rt_i) / np.log(rt_o / rt_i)

# Outer conductance: casing + cement in series
R_ca  = np.log(rca_o/rca_i) / (2*np.pi*kca)
R_ce  = np.log(rce  /rca_o) / (2*np.pi*kce)
Gext  = 1.0 / (R_ca + R_ce)

# Cement inner-wall temperature coefficients (for limit check)
alpha_cao = 1.0 - Gext*R_ca   # weight on T_annulus
beta_cao  =       Gext*R_ca   # weight on T_bw

print(f"Annulus area  Aa   = {Aa:.4e} m²")
print(f"Tubing  area  Atb  = {Atb:.4e} m²")
print(f"Hydraulic Dh_a     = {Dh_a*100:.2f} cm   Dh_tb = {Dh_tb*100:.2f} cm")
print(f"Outer conductance  Gext = {Gext:.2f} W/(m·K)")

def conductances(kt):
    """Annulus-tubing conductances for a given tubing wall conductivity kt."""
    Gat  = 1./(np.log(rt_o/rlm)/(2*np.pi*kt) + 1./(2*np.pi*rt_o))   # placeholder ha
    Gtbt = 1./(1./(2*np.pi*rt_i) + np.log(rlm/rt_i)/(2*np.pi*kt))   # placeholder htb
    # Note: ha, htb are multiplied in at NTU computation — see compute_NTUs()
    return kt   # conductivity is passed through; NTU functions handle convection

print(f"\nTubing conductivity:")
print(f"  Bare steel:  kt = {kt_bare} W/(m·K)")
print(f"  Insulated:   kt = {kt_insulated} W/(m·K)  (VIT equivalent)")


## 3 · Fluid Properties (CoolProp — temperature-dependent)

In [ ]:
def water_props(T_C, P_Pa=P_op):
    """Water properties at T_C [°C], P_Pa [Pa] via CoolProp."""
    if COOLPROP_OK:
        T_K = T_C + 273.15
        return {
            'rho': PropsSI('D',       'T',T_K,'P',P_Pa,'Water'),
            'cp' : PropsSI('C',       'T',T_K,'P',P_Pa,'Water'),
            'mu' : PropsSI('V',       'T',T_K,'P',P_Pa,'Water'),
            'kf' : PropsSI('L',       'T',T_K,'P',P_Pa,'Water'),
            'Pr' : PropsSI('Prandtl', 'T',T_K,'P',P_Pa,'Water'),
        }
    else:
        # Fallback at 60°C
        return {'rho':983.,'cp':4185.,'mu':4.67e-4,'kf':0.651,'Pr':3.00}

# Evaluate at mean fluid temperature (injection + expected return midpoint)
T_mean_est = (Tin + 85.) / 2.   # ~55°C initial estimate
props = water_props(T_mean_est)

print(f"Fluid properties at T_mean ≈ {T_mean_est:.0f}°C:")
for k,v in props.items():
    print(f"  {k:5s} = {v:.4g}")

if COOLPROP_OK:
    T_sat_160 = PropsSI('P','T',160+273.15,'Q',0,'Water')/1e5
    print(f"\nSaturation P at 160°C = {T_sat_160:.2f} bar  "
          f"(P_op = {P_op/1e5:.0f} bar → liquid ✓)")


## 4 · Convective Correlations and NTU Functions

In [ ]:
def gnielinski_h(v, Dh, props, Re_clamp=3100.):
    """Gnielinski/Petukhov h [W/(m²·K)]."""
    Re = props['rho'] * max(abs(v),1e-9) * Dh / props['mu']
    Re = max(Re, Re_clamp)
    fD = (0.790*np.log(Re) - 1.64)**(-2)
    Nu = (fD/8)*(Re-1000)*props['Pr'] / (
         1 + 12.7*np.sqrt(fD/8)*(props['Pr']**(2/3)-1))
    return Nu * props['kf'] / Dh


def compute_NTUs(va, Lbh, props, kt=kt_bare):
    """
    NTUeff_at (annulus↔tubing) and NTUa_bw (annulus↔formation wall).
    kt : tubing wall conductivity — kt_bare or kt_insulated.
    Returns (NTUeff_at, NTUa_bw, mdot_a).
    """
    mdot_a = props['rho'] * Aa * va
    vtb    = va * Aa / Atb
    ha     = gnielinski_h(va,  Dh_a,  props)
    htb    = gnielinski_h(vtb, Dh_tb, props)

    Gat  = 1./(np.log(rt_o/rlm)/(2*np.pi*kt) + 1./(ha *2*np.pi*rt_o))
    Gtbt = 1./(1./(htb*2*np.pi*rt_i)          + np.log(rlm/rt_i)/(2*np.pi*kt))

    NTUeff_at = Lbh/(mdot_a*props['cp']) / (1/Gat + 1/Gtbt)
    NTUa_bw   = Gext*Lbh / (mdot_a*props['cp'])
    return NTUeff_at, NTUa_bw, mdot_a


# ── Impact of insulation on NTUeff_at ─────────────────────────────────────
print(f"NTU comparison at Lbh={Lbh_default:.0f}m, va=0.05 m/s:")
print(f"{'Case':<20} {'NTUat':>8} {'NTUbw':>8}  Interpretation")
print("-"*58)
for label, kt in [('Bare steel',kt_bare),('Insulated VIT',kt_insulated)]:
    at, bw, _ = compute_NTUs(0.05, Lbh_default, props, kt)
    print(f"{label:<20} {at:>8.3f} {bw:>8.3f}  "
          f"{'Short-circuit SIGNIFICANT' if at>1 else 'Short-circuit suppressed'}")


## 5 · GITT Formation Model — Long-Term Setup
**Key difference from TES solver:** Rfar = 200 m to capture 20-year thermal diffusion.
The eigenvalue spectrum shifts to much smaller β₁ (longer time constants).


In [ ]:
def gitt_char_eq(beta):
    return y1(beta*rce)*j0(beta*Rfar) - j1(beta*rce)*y0(beta*Rfar)

def compute_eigenvalues(n_eig, beta_max=200., n_scan=800_000):
    betas = np.linspace(1e-6, beta_max, n_scan)
    fvals = np.vectorize(gitt_char_eq)(betas)
    roots = []
    for i in range(len(betas)-1):
        if fvals[i]*fvals[i+1] < 0:
            root = brentq(gitt_char_eq, betas[i], betas[i+1], xtol=1e-14)
            roots.append(root)
            if len(roots) == n_eig: break
    if len(roots) < n_eig:
        raise ValueError(f"Only {len(roots)}/{n_eig} eigenvalues found.")
    return np.array(roots)

def psi(beta, r):
    return y1(beta*rce)*j0(beta*r) - j1(beta*rce)*y0(beta*r)

def compute_norms(betas, Nr=6000):
    r = np.linspace(rce, Rfar, Nr)
    return np.array([np.trapezoid(r*psi(b,r)**2, r) for b in betas])

print(f"Computing {Neig} GITT eigenvalues  (Rfar = {Rfar:.0f} m) ...")
betas_arr  = compute_eigenvalues(Neig)
norms_arr  = compute_norms(betas_arr)
psi_rce    = psi(betas_arr, rce)
inv_wts    = psi_rce / norms_arr
forc_coeff = psi_rce / (2*np.pi*ke*betas_arr**2)
tau_arr    = 1./(alpha_e*betas_arr**2)

print(f"  β₁  = {betas_arr[0]:.6f} m⁻¹   τ₁  = {tau_arr[0]/86400/365:.1f} years")
print(f"  β₆₀ = {betas_arr[-1]:.4f} m⁻¹   τ₆₀ = {tau_arr[-1]/3600:.1f} h")
print("GITT setup complete ✓")

# Verify τ₁ >> simulation horizon
assert tau_arr[0] > n_years*365*86400*2,     f"τ₁={tau_arr[0]/86400/365:.1f}yr < 2×horizon — increase Rfar"
print(f"  τ₁ = {tau_arr[0]/86400/365:.0f} yr  >>  {n_years} yr horizon ✓")


## 6 · Fluid BVP — Continuous DBHE Flow
**Simplified vs TES solver:** single continuous flow direction only.
Cool water down annulus (U=+1), hot water up tubing. No discharge phase.


In [ ]:
def build_Phi_Psi(NTUeff_at, NTUa_bw, dz_star):
    A   = np.array([[-(NTUeff_at+NTUa_bw),  NTUeff_at],
                    [-NTUeff_at,              NTUeff_at]])
    Phi = expm(A*dz_star)
    Psi = np.linalg.solve(A, Phi - np.eye(2))
    return Phi, Psi


def solve_dbhe_bvp(NTUeff_at, NTUa_bw, theta_bw_cc, NZ, theta_a_in=0.):
    """
    DBHE fluid BVP: cool water injected down annulus (θa(0) = theta_a_in ≈ 0).
    Returns upward return temperature profile; θtb(0) is the surface return.

    With insulated tubing: NTUeff_at ≈ 0 → tubing thermally decoupled.
    With bare tubing:      NTUeff_at >> 0 → short-circuit reduces θtb(0).
    """
    dz    = 1./NZ
    Phi, Psi = build_Phi_Psi(NTUeff_at, NTUa_bw, dz)

    yH    = np.zeros((NZ+1, 2));  yH[0] = [0., 1.]
    yP    = np.zeros((NZ+1, 2));  yP[0] = [theta_a_in, 0.]

    for k in range(NZ):
        b_k      = np.array([NTUa_bw*theta_bw_cc[k], 0.])
        yH[k+1]  = Phi @ yH[k]
        yP[k+1]  = Phi @ yP[k] + Psi @ b_k

    # Turnaround: θtb(NZ) = θa(NZ)
    s = (yP[NZ,1] - yP[NZ,0]) / (yH[NZ,0] - yH[NZ,1])
    theta = s*yH + yP
    return theta[:,0], theta[:,1]   # θa(z*), θtb(z*)


# ── Quick BVP test ────────────────────────────────────────────────────────
theta_bw_test = 0.6*np.ones(NZ)   # dimensionless: θ=(T-Tin)/dT_scale
dT_scale = 100.                    # reference ΔT [K]

fig, axes = plt.subplots(1,2, figsize=(10,3.5), sharey=True)
z_star = np.linspace(0,1,NZ+1)
for ax, label, kt in zip(axes,
        ['Case A — Bare tubing','Case B — Insulated tubing'],
        [kt_bare, kt_insulated]):
    at, bw, _ = compute_NTUs(0.05, Lbh_default, props, kt)
    ta, ttb   = solve_dbhe_bvp(at, bw, theta_bw_test, NZ)
    ax.plot(ta*dT_scale + Tin,  z_star*Lbh_default, label='Annulus (↓)')
    ax.plot(ttb*dT_scale + Tin, z_star*Lbh_default, label='Tubing  (↑)', ls='--')
    ax.axvline(Tin + theta_bw_test[0]*dT_scale, color='grey',
               ls=':', lw=0.8, label='T_bw')
    ax.set_xlabel('T [°C]'); ax.set_title(label)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.invert_yaxis()
    surf_T = Tin + ttb[0]*dT_scale
    ax.axvline(78., color='r', ls='--', lw=0.9, label='LiBr min 78°C')
    print(f"{label}: Tret = {surf_T:.1f}°C  "
          f"({'✓ Above LiBr threshold' if surf_T>=78 else '✗ Below LiBr threshold'})")

axes[0].set_ylabel('Depth [m]')
plt.suptitle('BVP test — θbw = const = 0.6, Lbh=2500m, va=0.05m/s', fontweight='bold')
plt.tight_layout(); plt.show()


## 7 · LiBr Single-Effect Absorption Chiller Model
Characteristic equation method (Ziegler & Treeck, fitted to manufacturer data).
Valid range: T_gen = 75–110°C, T_cond = 28–40°C, T_evap = 5–12°C.


In [ ]:
# ── Chiller operating temperatures ────────────────────────────────────────
T_cond = 35.0   # condenser temperature [°C]  (cooling tower, Gulf Coast summer)
T_evap =  7.0   # evaporator temperature [°C] (chilled water supply)
T_abs  = 37.0   # absorber temperature [°C]   (≈ T_cond + 2)

# ── Characteristic equation method (ΔΔT approach) ────────────────────────
# COP_cooling = a + b·ΔΔT
# ΔΔT = T_gen + T_evap - f_char·(T_cond + T_abs)
# Coefficients fitted to Carrier/Trane single-effect LiBr product data
a_char  =  0.395
b_char  =  0.00315   # [1/K]
f_char  =  1.00

T_libr_min = 78.0    # minimum generator temperature for stable operation [°C]
T_libr_max = 110.0   # maximum generator temperature (crystallisation risk above) [°C]

def COP_LiBr(T_gen, T_cond=T_cond, T_evap=T_evap, T_abs=T_abs):
    """
    Single-effect LiBr COP_cooling via characteristic equation.
    Returns 0 if T_gen outside viable range.
    """
    if T_gen < T_libr_min or T_gen > T_libr_max:
        return 0.0
    ddt = T_gen + T_evap - f_char*(T_cond + T_abs)
    return max(0., a_char + b_char*ddt)

def cold_output(mdot, cp, T_ret, T_in=Tin):
    """Cold production rate [W] from wellbore return stream."""
    Q_geo = mdot * cp * max(T_ret - T_in, 0.)
    cop   = COP_LiBr(T_ret)
    return Q_geo * cop, Q_geo, cop

# ── COP curve ─────────────────────────────────────────────────────────────
T_gen_arr = np.linspace(70, 115, 200)
COP_arr   = [COP_LiBr(T) for T in T_gen_arr]

fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(T_gen_arr, COP_arr, 'steelblue', lw=2)
ax.axvline(T_libr_min, color='r', ls='--', lw=1, label=f'Min T_gen = {T_libr_min}°C')
ax.axvline(T_libr_max, color='orange', ls='--', lw=1, label=f'Max T_gen = {T_libr_max}°C')
ax.fill_betweenx([0,0.9], T_libr_min, T_libr_max, alpha=0.08, color='green',
                 label='Viable operating band')
ax.set_xlabel('Generator temperature T_gen = T_ret [°C]')
ax.set_ylabel('COP_cooling [-]')
ax.set_title('Single-effect LiBr chiller — characteristic equation COP')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim(0, 0.9)
plt.tight_layout(); plt.show()

print(f"COP at T_ret = 80°C: {COP_LiBr(80.):.3f}")
print(f"COP at T_ret = 90°C: {COP_LiBr(90.):.3f}")
print(f"COP at T_ret = 100°C: {COP_LiBr(100.):.3f}")


## 8 · Coupled DBHE–Chiller Solver (Monthly Time Steps, 20-Year Horizon)

In [ ]:
def T_inf_z(z_arr):
    return T_surf + Gg * z_arr

def run_dbhe(Lbh, va, kt=kt_bare,
             n_months=n_months, verbose=True):
    """
    Long-term DBHE solver. Monthly GITT updates.
    Each month: fluid BVP solved at monthly-average Tbw;
    formation updated with effective daily-average heat flux
    (accounts for f_op operating fraction).

    Parameters
    ----------
    Lbh      : depth [m]
    va       : annulus velocity [m/s]
    kt       : tubing conductivity — kt_bare or kt_insulated
    n_months : simulation duration

    Returns
    -------
    dict with monthly time series of Tret, Q_cold, COP, etc.
    """
    label = 'Bare' if kt > 1. else 'Insulated'

    # Update fluid props at estimated mean temperature
    T_mean = (Tin + (T_surf + Gg*Lbh*0.6)) / 2.
    pr     = water_props(min(T_mean, 95.))

    NTUat, NTUbw, mdot = compute_NTUs(va, Lbh, pr, kt)
    if verbose:
        print(f"[{label}] Lbh={Lbh:.0f}m  va={va:.3f}m/s  "
              f"NTUat={NTUat:.3f}  NTUbw={NTUbw:.3f}  mdot={mdot:.3f}kg/s")

    # ── GITT initialisation ──────────────────────────────────────────────
    dz_m  = Lbh / NZ
    z_cc  = (np.arange(NZ)+0.5)*dz_m
    U_bar = np.zeros((NZ, Neig))
    Tbw   = T_inf_z(z_cc).copy()
    dT_sc = max(Tbw.max() - Tin, 1.)   # dimensionless scale [K]

    # Monthly decay factors
    decay_mo = np.exp(-alpha_e*betas_arr**2*dt_month)

    def _gitt_step(U_bar, Tbw, T_a_cc, decay):
        q_bw  = Gext*(T_a_cc - Tbw)
        # Effective heat flux = actual flux × operating fraction
        q_eff = q_bw * f_op
        U_bar = (U_bar*decay[np.newaxis,:]
                 + forc_coeff[np.newaxis,:]*(1-decay[np.newaxis,:])
                 * q_eff[:,np.newaxis])
        Tbw_new = T_inf_z(z_cc) + (U_bar*inv_wts[np.newaxis,:]).sum(axis=1)
        return U_bar, Tbw_new

    def _cc(theta_edge):
        return 0.5*(theta_edge[:-1]+theta_edge[1:])

    # ── Output arrays ────────────────────────────────────────────────────
    Tret_hist   = np.zeros(n_months)
    Qcold_hist  = np.zeros(n_months)
    Qgeo_hist   = np.zeros(n_months)
    COP_hist    = np.zeros(n_months)
    viable_hist = np.zeros(n_months, dtype=bool)

    for mo in range(n_months):
        theta_bw = (Tbw - Tin) / dT_sc

        # BVP — cool water down, hot return up
        ta, ttb = solve_dbhe_bvp(NTUat, NTUbw, theta_bw, NZ,
                                  theta_a_in=0.)

        T_a_cc  = Tin + _cc(ta)  * dT_sc
        T_ret   = Tin + ttb[0]   * dT_sc   # surface return temperature

        # GITT formation update (monthly)
        U_bar, Tbw = _gitt_step(U_bar, Tbw, T_a_cc, decay_mo)

        # Chiller performance
        Q_cold, Q_geo, cop = cold_output(mdot, pr['cp'], T_ret)

        Tret_hist[mo]   = T_ret
        Qgeo_hist[mo]   = Q_geo
        Qcold_hist[mo]  = Q_cold
        COP_hist[mo]    = cop
        viable_hist[mo] = T_ret >= T_libr_min

        if verbose and (mo % 12 == 0 or mo == n_months-1):
            yr = (mo+1)/12
            print(f"  Year {yr:5.1f}: Tret={T_ret:.1f}°C  "
                  f"Q_geo={Q_geo/1e3:.1f}kW  "
                  f"Q_cold={Q_cold/1e3:.1f}kW  "
                  f"COP={cop:.3f}  "
                  f"{'✓' if T_ret>=T_libr_min else '✗ END-OF-LIFE'}")

    # End-of-life: first month Tret < T_libr_min
    eol_months = np.argmax(~viable_hist) if not viable_hist.all() else n_months
    eol_years  = eol_months / 12.

    return dict(
        Tret=Tret_hist, Qgeo=Qgeo_hist, Qcold=Qcold_hist,
        COP=COP_hist, viable=viable_hist,
        eol_years=eol_years, label=label,
        Lbh=Lbh, va=va, kt=kt,
        mdot=mdot, NTUat=NTUat, NTUbw=NTUbw,
        total_cold_GWh = Qcold_hist.sum()*dt_month*f_op / 3.6e9
    )


## 9 · Demo Run — Bare vs Insulated, Lbh = 2500 m

In [ ]:
print("="*60)
print("Case A: Bare steel tubing")
print("="*60)
res_bare = run_dbhe(Lbh=2500., va=0.05, kt=kt_bare, verbose=True)

print("\n" + "="*60)
print("Case B: Insulated (VIT) tubing")
print("="*60)
res_ins  = run_dbhe(Lbh=2500., va=0.05, kt=kt_insulated, verbose=True)


### 9.1 · Results — Bare vs Insulated Comparison

In [ ]:
t_yr = np.arange(1, n_months+1) / 12.

fig = plt.figure(figsize=(14,10))
gs  = gridspec.GridSpec(2,3, figure=fig, hspace=0.40, wspace=0.35)

colors = {'Bare':'tomato', 'Insulated':'steelblue'}

# (a) Tret over 20 years
ax1 = fig.add_subplot(gs[0,0])
for res in [res_bare, res_ins]:
    ax1.plot(t_yr, res['Tret'], color=colors[res['label']],
             label=res['label'], lw=1.8)
ax1.axhline(T_libr_min, color='k', ls='--', lw=1,
            label=f'LiBr min {T_libr_min}°C')
ax1.axhline(T_libr_max, color='grey', ls=':', lw=1,
            label=f'LiBr max {T_libr_max}°C')
ax1.set_xlabel('Year'); ax1.set_ylabel('T_ret [°C]')
ax1.set_title('Surface return temperature')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# (b) Q_cold over 20 years
ax2 = fig.add_subplot(gs[0,1])
for res in [res_bare, res_ins]:
    ax2.plot(t_yr, res['Qcold']/1e3, color=colors[res['label']],
             label=res['label'], lw=1.8)
ax2.set_xlabel('Year'); ax2.set_ylabel('Q_cold [kW]')
ax2.set_title('Cold production rate')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# (c) COP over 20 years
ax3 = fig.add_subplot(gs[0,2])
for res in [res_bare, res_ins]:
    ax3.plot(t_yr, res['COP'], color=colors[res['label']],
             label=res['label'], lw=1.8)
ax3.set_xlabel('Year'); ax3.set_ylabel('COP_cooling [-]')
ax3.set_title('Chiller COP')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# (d) Tret degradation rate (annual decline)
ax4 = fig.add_subplot(gs[1,0])
for res in [res_bare, res_ins]:
    dTret = -np.diff(res['Tret'].reshape(-1,12).mean(axis=1))
    ax4.bar(np.arange(1,n_years)+0.5*(1 if res['label']=='Bare' else -1)*0.35,
            dTret, 0.35, color=colors[res['label']], alpha=0.8,
            label=res['label'])
ax4.set_xlabel('Year'); ax4.set_ylabel('ΔT_ret per year [K]')
ax4.set_title('Annual T_ret decline rate')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3, axis='y')

# (e) Cumulative cold delivered
ax5 = fig.add_subplot(gs[1,1])
for res in [res_bare, res_ins]:
    cum = np.cumsum(res['Qcold'])*dt_month*f_op/3.6e9
    ax5.plot(t_yr, cum, color=colors[res['label']],
             label=res['label'], lw=1.8)
ax5.set_xlabel('Year'); ax5.set_ylabel('Cumulative Q_cold [GWh]')
ax5.set_title('Total cold energy delivered')
ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

# (f) Summary table
ax6 = fig.add_subplot(gs[1,2])
ax6.axis('off')
def fmt(res):
    eol = f"{res['eol_years']:.1f} yr" if res['eol_years']<n_years else f">{n_years} yr"
    return (
        f"Case: {res['label']}
"
        f"Lbh     = {res['Lbh']:.0f} m
"
        f"va      = {res['va']:.3f} m/s
"
        f"NTUat   = {res['NTUat']:.3f}
"
        f"─────────────────
"
        f"Tret(t=0)= {res['Tret'][0]:.1f}°C
"
        f"Tret(20yr)={res['Tret'][-1]:.1f}°C
"
        f"End-of-life: {eol}
"
        f"Total cold: {res['total_cold_GWh']:.2f} GWh
"
        f"─────────────────
"
        f"COP(t=0) = {res['COP'][0]:.3f}
"
        f"COP(20yr)= {res['COP'][-1]:.3f}"
    )
ax6.text(0.05,0.97, fmt(res_bare)+"

"+fmt(res_ins),
         transform=ax6.transAxes, va='top', fontsize=8.5,
         family='monospace',
         bbox=dict(boxstyle='round',facecolor='#f5f5f5',alpha=0.9))
ax6.set_title('Performance summary')

plt.suptitle(
    f'DBHE Absorption Cooling — Onshore S. Louisiana  |  '
    f'Lbh={res_bare["Lbh"]:.0f}m  va={res_bare["va"]:.3f}m/s  '
    f'Gg={Gg*1000:.0f} mK/m  |  {n_years}-year horizon',
    fontsize=11, fontweight='bold')
plt.show()


## 10 · Parametric Sweep — Depth and Flow Rate

In [ ]:
Lbh_sweep = np.array([1500., 2000., 2500., 3000., 3500.])
va_sweep  = np.array([0.02, 0.05, 0.10, 0.20])

# Storage: [case, va, Lbh] where case 0=bare, 1=insulated
Tret0  = np.full((2,len(va_sweep),len(Lbh_sweep)), np.nan)  # initial Tret
Tret20 = np.full_like(Tret0, np.nan)                          # 20-yr Tret
Qc0    = np.full_like(Tret0, np.nan)                          # initial Q_cold [kW]
EOL    = np.full_like(Tret0, np.nan)                          # end-of-life [yr]
TotGWh = np.full_like(Tret0, np.nan)                          # total cold GWh

kt_cases = [kt_bare, kt_insulated]
labels   = ['Bare','Insulated']

print(f"{'Case':<12}{'va':>6}{'Lbh':>7}  "
      f"{'Tret_0':>8}{'Tret_20':>9}{'Qc_0[kW]':>10}{'EOL[yr]':>9}{'GWh':>7}")
print("-"*72)
for ci, (kt, label) in enumerate(zip(kt_cases, labels)):
    for vi, va in enumerate(va_sweep):
        for li, Lbh in enumerate(Lbh_sweep):
            r = run_dbhe(Lbh, va, kt=kt, n_months=n_months, verbose=False)
            Tret0[ci,vi,li]  = r['Tret'][0]
            Tret20[ci,vi,li] = r['Tret'][-1]
            Qc0[ci,vi,li]    = r['Qcold'][0]/1e3
            EOL[ci,vi,li]    = r['eol_years']
            TotGWh[ci,vi,li] = r['total_cold_GWh']
            eol_s = f"{r['eol_years']:.1f}" if r['eol_years']<n_years else f">{n_years}"
            print(f"{label:<12}{va:>6.2f}{Lbh:>7.0f}  "
                  f"{r['Tret'][0]:>8.1f}{r['Tret'][-1]:>9.1f}"
                  f"{r['Qcold'][0]/1e3:>10.1f}{eol_s:>9}{r['total_cold_GWh']:>7.2f}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15,9))

# Fix va for heatmaps (index 1 = 0.05 m/s)
vi_plot = 1

plot_data = [
    (Tret0,  'Initial T_ret [°C]',        'RdYlGn',   False),
    (Tret20, '20-yr T_ret [°C]',           'RdYlGn',   False),
    (Qc0,    'Initial Q_cold [kW]',        'viridis',  False),
    (EOL,    'End-of-life [yr]',           'viridis',  False),
    (TotGWh, 'Total cold over 20yr [GWh]', 'viridis',  False),
]

for row, (ax_row) in enumerate(axes):
    for col, ax in enumerate(ax_row):
        idx = row*3 + col
        if idx >= len(plot_data): ax.axis('off'); continue
        data, title, cmap, _ = plot_data[idx]
        # Show bare (ci=0) and insulated (ci=1) side by side via subplots
        # Here plot difference (insulated - bare) for last panel
        if idx == 4:
            d = TotGWh[1] - TotGWh[0]
            im = ax.pcolormesh(Lbh_sweep, va_sweep*100, d[:,vi_plot,:] if d.ndim==3
                               else d, shading='nearest', cmap='PiYG')
            plt.colorbar(im, ax=ax)
            ax.set_title('Insulated gain [GWh]')
        else:
            for ci, (ls, lbl) in enumerate([('-','Bare'),('--','Insulated')]):
                d = plot_data[idx][0][ci, vi_plot, :]
                ax.plot(Lbh_sweep, d, ls+('o' if ci==0 else 's'),
                        label=lbl, lw=1.8)
            ax.axhline(T_libr_min if 'T_ret' in title else 0,
                       color='r', ls=':', lw=0.8)
            ax.set_title(f'{title}  (va={va_sweep[vi_plot]:.2f}m/s)')
            ax.legend(fontsize=8)
        ax.set_xlabel('Depth [m]')
        ax.grid(True, alpha=0.3)

plt.suptitle(
    f'DBHE Parametric Sweep — Onshore S. Louisiana  |  '
    f'Gg={Gg*1000:.0f} mK/m  |  {n_years}-yr horizon  |  '
    f'f_op={f_op:.0%}',
    fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()
